# Assignment 10 - From Images to Short Textual Descriptions
### A Vision Transformer from Scratch with Qwen2.5-0.5B-Instruct

**Duration:** 3 hours  |  **Maximum marks:** 50  |  **Seed:** 42

**Student notebook :- complete the `TODO` sections.** Read the supplied assignment PDF for the full instructions and marking policy. This notebook provides imports, paths, configuration, and task cells **without assessed implementations**.

**System:** `image → your ViT → top-two class evidence → frozen text-only Qwen → one-sentence description`. Qwen does **not** see image pixels. Do not claim to identify colours, actions, counts, or backgrounds from class predictions.

## Before starting

Extract the supplied data/model archives such that the structure, relative to the folder containing this notebook, is:

```text
Assignment_10/
├── Assignment_10_Starter.ipynb
├── data/
│   └── cifar10_lab.npz
└── models/
    └── Qwen2.5-0.5B-Instruct/
        ├── config.json
        ├── model.safetensors
        └── ... (tokenizer and other model files)
```

**Rules:** Work completely offline. No model downloads, internet, pretrained ViT, `nn.MultiheadAttention`, `nn.TransformerEncoder`, `nn.TransformerEncoderLayer`, or ready-made attention functions. Train **only** the ViT; Qwen remains frozen. Keep the provided data split and test order.

Run cells in order. If an extraction created an extra nested directory, move the extracted directory to match the paths above.

In [ ]:
# Common imports (provided; not graded)
import os
import time
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

# Do not permit remote Hugging Face retrievals.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()  # Open/run this notebook from the Assignment_10 folder.
DATA_PATH = ROOT / "data" / "cifar10_lab.npz"
MODEL_PATH = ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = ROOT

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Tested-setting convention for this lab: FP16 on CUDA, FP32 otherwise.
QWEN_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

IMAGE_SIZE = 32
IN_CHANNELS = 3
PATCH_SIZE = 4
EMBED_DIM = 64
NUM_BLOCKS = 2
NUM_HEADS = 4
MLP_DIM = 128
NUM_CLASSES = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 5
TRAINING_TIME_LIMIT_SECONDS = 12 * 60
GRAD_CLIP_NORM = 1.0
DROPOUT = 0.0

print("Python seed:", SEED)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("Qwen dtype:", QWEN_DTYPE)
print("Data exists:", DATA_PATH.is_file(), "→", DATA_PATH)
print("Qwen config exists:", (MODEL_PATH / "config.json").is_file(), "→", MODEL_PATH)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset missing: {DATA_PATH}. Check ZIP extraction/folder names.")
if not (MODEL_PATH / "config.json").is_file():
    raise FileNotFoundError(f"Qwen model missing: {MODEL_PATH}. Check ZIP extraction/folder names.")

## Question 1 - Dataset Inspection and Input Pipeline (10 marks)
### Task 1.1 - Inspect the supplied splits (4 marks)
Load the seven arrays from `cifar10_lab.npz`. Print each split's shape, min/max pixel range, and counts for each class. Verify **10,000 training**, **1,000 validation**, and **1,000 test** images. **Do not change the splits.**

In [ ]:
# Task 1.1 (4 marks)
with np.load(DATA_PATH) as data:
    x_train, y_train = data["x_train"], data["y_train"]
    x_val, y_val = data["x_val"], data["y_val"]
    x_test, y_test = data["x_test"], data["y_test"]
    class_names = [str(name) for name in data["class_names"]]

splits = {
    "train": (x_train, y_train, 10_000),
    "val": (x_val, y_val, 1_000),
    "test": (x_test, y_test, 1_000),
}
print("Class names:", class_names)
for split_name, (images, labels, expected_size) in splits.items():
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    print(f"\n{split_name}: images={images.shape} ({images.dtype}), labels={labels.shape}")
    print(f"  pixel range: min={images.min()}, max={images.max()}")
    print("  class counts:", dict(zip(class_names, counts.tolist())))
    assert images.shape == (expected_size, IMAGE_SIZE, IMAGE_SIZE, IN_CHANNELS)
    assert labels.shape == (expected_size,)
assert len(class_names) == NUM_CLASSES
print("\nRequired split sizes verified: 10,000 train, 1,000 validation, 1,000 test.")

### Task 1.2 -  Prepare tensors and verify a batch (6 marks)

Create a PyTorch `Dataset` and three `DataLoader`s. Convert images from `(32, 32, 3)` `uint8` to `(3, 32, 32)` `float32`; normalize with **`x_normalized = 2 * (x / 255) - 1`**. Use integer (`long`) class labels for cross-entropy. Shuffle **training only** and use **`num_workers=0`**. Print one batch's shapes and display **four training images** with class names after reversing normalization. No augmentation.

In [ ]:
# Task 1.2 (6 marks)
class MyDataset(Dataset):
    def __init__(self, x, y):
        super().__init__()
        assert x.shape[0] == y.shape[0]
        self.x = x
        self.y = y

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        image = self.x[idx].transpose(2, 0, 1).astype(np.float32)
        image = 2 * (image / 255.0) - 1
        return torch.from_numpy(image), torch.tensor(int(self.y[idx]), dtype=torch.long)

train_dataset = MyDataset(x_train, y_train)
val_dataset = MyDataset(x_val, y_val)
test_dataset = MyDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch_images, batch_labels = next(iter(train_loader))
print("Image batch:", batch_images.shape, batch_images.dtype)
print("Label batch:", batch_labels.shape, batch_labels.dtype)
print(f"Normalized range: {batch_images.min().item():.3f} to {batch_images.max().item():.3f}")

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    display_image = ((batch_images[i] + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(display_image)
    ax.set_title(class_names[batch_labels[i].item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Question 2 - Vision Transformer from Scratch (15 marks)

**Required architecture:** 4×4 non-overlapping patches; 64-dimensional embeddings; a learnable `[CLS]` token; learnable positional embeddings; **two pre-normalization** encoder blocks; four attention heads; `64 → 128 → 64` GELU MLP; final LayerNorm and linear 10-class head. Random initialization, dropout 0. Implement attention using elementary PyTorch ops (`nn.Linear`, tensor multiplication, `torch.softmax` etc.), **not** ready-made attention functions.

### Task 2.1 - Patch embedding and positional information (4 marks)

Implement `PatchEmbedding`. Either use `nn.Conv2d(3,64,kernel_size=4,stride=4)` then flatten/transposition, or extract patches and project with a linear layer. Compute `N`, print patch token shape, prepend the learned class token and add learned positional embeddings of shape `(1, N+1, 64)`. Print the resulting sequence shape for **two** dummy images. Explain positional embeddings in one sentence.

Hint for your shape check: 32/4 = 8 patches per side, so `N = 64` and the sequence contains 65 tokens including `[CLS]`.

In [ ]:
# Task 2.1 (4 marks)
class PatchEmbedding(nn.Module):
    """Project non-overlapping patches to tokens of shape (B, N, EMBED_DIM)."""
    def __init__(self):
        super().__init__()
        self.num_patches = (IMAGE_SIZE // PATCH_SIZE) ** 2
        self.proj = nn.Conv2d(IN_CHANNELS, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)

    def forward(self, images):
        return self.proj(images).flatten(2).transpose(1, 2)


class TokenEmbedding(nn.Module):
    """Patch tokens + learnable [CLS] token + learnable positional embeddings."""
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        num_tokens = self.patch_embedding.num_patches + 1
        self.cls_token = nn.Parameter(torch.empty(1, 1, EMBED_DIM))
        self.pos_embedding = nn.Parameter(torch.empty(1, num_tokens, EMBED_DIM))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

    def forward(self, images):
        patch_tokens = self.patch_embedding(images)
        cls_tokens = self.cls_token.expand(patch_tokens.shape[0], -1, -1)
        return torch.cat((cls_tokens, patch_tokens), dim=1) + self.pos_embedding


dummy_images = torch.rand(2, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
token_embedding_demo = TokenEmbedding()
patch_tokens_demo = token_embedding_demo.patch_embedding(dummy_images)
dummy_sequence = token_embedding_demo(dummy_images)
print("N =", token_embedding_demo.patch_embedding.num_patches)
print("Patch token shape:", patch_tokens_demo.shape)
print("Positional embedding shape:", token_embedding_demo.pos_embedding.shape)
print("Sequence shape for two images:", dummy_sequence.shape)

**Task 2.1 written answer - why positional embeddings are needed:**

> Self-attention alone does not encode token order, so learned positional embeddings give each patch information about its location in the image.

### Task 2.2 - Multi-head self-attention (7 marks)

Implement trainable `Q`, `K`, `V`, and output projections (one fused QKV projection is allowed). Split into `h=4` heads with `d_h=16`. Use **full bidirectional attention**, with no causal mask:

\[
A_j=\operatorname{softmax}\left(\frac{Q_jK_j^\top}{\sqrt{d_h}}\right),\qquad H_j=A_jV_j.
\]

Concatenate head outputs and apply the final projection. Show the shapes of **Q**, the **attention matrix**, and the **output** for a dummy batch. Check that attention rows sum to approximately one.

In [ ]:
# Task 2.2 (7 marks)
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim=EMBED_DIM, num_heads=NUM_HEADS, debug=False):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.debug = debug

    def _split_heads(self, x):
        batch_size, num_tokens, _ = x.shape
        return x.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

    def _merge_heads(self, x):
        batch_size, _, num_tokens, _ = x.shape
        return x.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.dim)

    def forward(self, x):
        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention = torch.softmax(scores, dim=-1)
        output = self.out_proj(self._merge_heads(torch.matmul(attention, v)))

        if self.debug:
            row_sums = attention.sum(dim=-1)
            print("Q shape (per head):", q.shape)
            print("Attention matrix shape:", attention.shape)
            print(f"Attention row sums: min={row_sums.min().item():.6f}, max={row_sums.max().item():.6f}")
            print("Output shape:", output.shape)
        return output


mha_demo = MultiHeadSelfAttention(debug=True)
with torch.no_grad():
    mha_demo(dummy_sequence)

### Task 2.3 - Encoder blocks and classification head (4 marks)

Use separate normalization layers and residual connections:

\[
U=X+\operatorname{MSA}(\operatorname{LN}_1(X)),\qquad
Y=U+\operatorname{MLP}(\operatorname{LN}_2(U)).
\]

The MLP is `Linear(64,128) → GELU → Linear(128,64)`. Stack two blocks. Apply final LayerNorm, select the CLS token and return class logits through `Linear(64,10)` (**no softmax before cross-entropy**). Print number of trainable parameters and check output shape `(2,10)`.

In [ ]:
# Task 2.3 (4 marks)
class EncoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(EMBED_DIM)
        self.ln2 = nn.LayerNorm(EMBED_DIM)
        self.msa = MultiHeadSelfAttention()
        self.mlp = nn.Sequential(
            nn.Linear(EMBED_DIM, MLP_DIM),
            nn.GELU(),
            nn.Linear(MLP_DIM, EMBED_DIM),
        )

    def forward(self, x):
        u = x + self.msa(self.ln1(x))
        return u + self.mlp(self.ln2(u))


class TinyViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = TokenEmbedding()
        self.blocks = nn.Sequential(*[EncoderBlock() for _ in range(NUM_BLOCKS)])
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, NUM_CLASSES)

    def forward(self, images):
        x = self.embedding(images)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.head(x[:, 0])  # raw logits from the CLS token; no softmax


model_demo = TinyViT()
trainable_params = sum(p.numel() for p in model_demo.parameters() if p.requires_grad)
with torch.no_grad():
    demo_logits = model_demo(dummy_images)
print("Trainable parameters:", trainable_params)
print("Logits shape for two images:", demo_logits.shape)
assert demo_logits.shape == (2, NUM_CLASSES)

## Question 3 - Training, Evaluation, and the Text Interface (15 marks)
### Task 3.1 - Train the visual classifier (6 marks)

Use cross-entropy and AdamW with supplied hyperparameters. Clip gradients to norm **1.0**. Use `train()` for training; `eval()` and no gradients for validation. Train **up to five epochs**, stopping **after the first completed epoch for which elapsed training time exceeds 12 minutes**, or after epoch five. Record completed epochs and wall-clock training time.

Each epoch: report **sample-weighted** average training loss, sample-weighted average validation loss, and validation accuracy. Save the lowest-validation-loss checkpoint as `best_vit.pt` **including or alongside its architecture configuration and class-name order**. Plot train/validation loss against completed epochs. Never use test-set results for tuning or selection.

In [ ]:
# Task 3.1 (6 marks)
CHECKPOINT_PATH = OUTPUT_DIR / "best_vit.pt"
ARCHITECTURE_CONFIG = {
    "image_size": IMAGE_SIZE,
    "in_channels": IN_CHANNELS,
    "patch_size": PATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "num_blocks": NUM_BLOCKS,
    "num_heads": NUM_HEADS,
    "mlp_dim": MLP_DIM,
    "num_classes": NUM_CLASSES,
    "dropout": DROPOUT,
}


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    device = next(model.parameters()).device
    total_loss = 0.0
    total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
    return total_loss / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    device = next(model.parameters()).device
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        batch_size = labels.size(0)
        total_loss += criterion(logits, labels).item() * batch_size
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total_samples += batch_size
    return total_loss / total_samples, total_correct / total_samples


def train(model, train_loader, val_loader, criterion, optimizer):
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    start_time = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        elapsed = time.perf_counter() - start_time

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f} | elapsed={elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "model_state_dict": model.state_dict(),
                "architecture_config": ARCHITECTURE_CONFIG,
                "class_names": class_names,
            }, CHECKPOINT_PATH)
            print(f"  Saved best checkpoint at epoch {epoch}")

        if elapsed > TRAINING_TIME_LIMIT_SECONDS:
            print("Stopping: the training time limit was exceeded.")
            break

    wall_clock_seconds = time.perf_counter() - start_time
    return history, wall_clock_seconds


seed_all(SEED)
model = TinyViT().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history, training_seconds = train(model, train_loader, val_loader, criterion, optimizer)
print(f"\nCompleted epochs: {len(history['epoch'])}")
print(f"Wall-clock training time: {training_seconds:.1f} s")

In [ ]:
plt.plot(history["epoch"], history["train_loss"], marker="o", label="Train loss")
plt.plot(history["epoch"], history["val_loss"], marker="s", label="Validation loss")
plt.xticks(history["epoch"])
plt.xlabel("Completed epoch")
plt.ylabel("Sample-weighted cross-entropy loss")
plt.title("TinyViT training and validation loss")
plt.legend()
plt.show()

### Task 3.2 - Evaluate the selected model (5 marks)

Reload the **best** checkpoint and evaluate all **1,000 test images** using `eval()` and no gradients. Report top-1 test accuracy. Show a named **10×10 confusion matrix** (rows = true class, columns = predicted class). Identify one frequent observed confusion, or state that none occurred. No assumed accuracy target.

In [ ]:
# Task 3.2 (5 marks) - reload the best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["architecture_config"] == ARCHITECTURE_CONFIG
assert checkpoint["class_names"] == class_names

best_model = TinyViT().to(DEVICE)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()
print(f"Loaded epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f}, "
      f"val_acc={checkpoint['val_acc']:.4f})")

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, loader):
    model.eval()
    device = next(model.parameters()).device
    matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)  # rows=true, columns=predicted
    for images, labels in loader:
        predictions = model(images.to(device)).argmax(dim=-1).cpu().numpy()
        for true_label, predicted_label in zip(labels.numpy(), predictions):
            matrix[true_label, predicted_label] += 1
    return matrix


conf_mat = compute_confusion_matrix(best_model, test_loader)
assert conf_mat.sum() == 1000
test_accuracy = np.trace(conf_mat) / conf_mat.sum()
print(f"Top-1 test accuracy: {test_accuracy:.4f} ({np.trace(conf_mat)}/{conf_mat.sum()})")

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(conf_mat, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Test-set confusion matrix")
threshold = conf_mat.max() / 2
for row in range(NUM_CLASSES):
    for col in range(NUM_CLASSES):
        color = "white" if conf_mat[row, col] > threshold else "black"
        ax.text(col, row, conf_mat[row, col], ha="center", va="center", color=color)
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

off_diagonal = conf_mat.copy()
np.fill_diagonal(off_diagonal, 0)
if off_diagonal.max() == 0:
    print("No confusion occurred on the test set.")
else:
    true_idx, predicted_idx = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
    print(f"Most frequent confusion: true '{class_names[true_idx]}' predicted as "
          f"'{class_names[predicted_idx]}' ({off_diagonal[true_idx, predicted_idx]} images).")

**Task 3.2 observation - frequent confusion (based on your own matrix):**

> The most frequent observed confusion is the one printed by the code cell above: **[copy the printed true/predicted classes and count here after running]**.

### Task 3.3 - Convert predictions into textual evidence (4 marks)

Implement `build_evidence(logits, class_names)` to apply softmax, take the two highest-scoring classes, and return **only** class names, scores rounded to 3 decimals, and the fixed uncertainty flag:

\[
\mathrm{uncertain}=(p_{\max}<0.50).
\]

Example format: `top1=cat; score1=0.420; top2=dog; score2=0.310; uncertain=true`.

Demonstrate on the **first five test images in original array order**; use the same five in Question 4. Do **not** input true labels, filenames, or test accuracy into this function or Qwen. These scores are not calibrated correctness probabilities, and two classes are alternatives-not evidence that both objects appear.

In [ ]:
# Task 3.3 (4 marks)
def build_evidence(logits, class_names):
    logits = logits.detach().squeeze()
    if logits.dim() != 1:
        raise ValueError("build_evidence expects logits for one image.")
    probabilities = torch.softmax(logits, dim=-1)
    top_scores, top_indices = torch.topk(probabilities, k=2)
    score1, score2 = top_scores.tolist()
    top1, top2 = [class_names[index] for index in top_indices.tolist()]
    uncertain = score1 < 0.50  # uses the unrounded maximum probability
    return (f"top1={top1}; score1={score1:.3f}; top2={top2}; score2={score2:.3f}; "
            f"uncertain={str(uncertain).lower()}")


def parse_evidence(evidence):
    return dict(part.split("=", 1) for part in evidence.split("; "))


# First five test images in original array order; logits are the only input to build_evidence.
NUM_SAMPLES = 5
best_model.eval()
first_five_images = torch.stack([test_dataset[i][0] for i in range(NUM_SAMPLES)]).to(DEVICE)
with torch.no_grad():
    first_five_logits = best_model(first_five_images).cpu()

evidences = [build_evidence(logits, class_names) for logits in first_five_logits]
for index, evidence in enumerate(evidences):
    print(f"Test image {index}: {evidence}")

**Task 3.3 written answer - why ViT features cannot be fed directly to a text-only Qwen model (1-2 sentences):**

> Text-only Qwen accepts token IDs mapped through its own word embeddings, whereas the 64-dimensional ViT features come from a different, unaligned space. Without a trained projection or multimodal fusion component, Qwen has no learned way to interpret those features, so the classifier output must be converted into text.

## Question 4 - Offline Description Generation and Analysis (10 marks)
### Task 4.1 - Load Qwen and generate descriptions (4 marks)

Load the tokenizer and frozen `AutoModelForCausalLM` from `MODEL_PATH` using **`local_files_only=True`**. Use the provided `DEVICE` / `QWEN_DTYPE` settings. Set `eval()`, freeze all model parameters, and generate one description at a time under `torch.inference_mode()`.

Build a **system + user** chat prompt using `tokenizer.apply_chat_template(..., add_generation_prompt=True)`; use Task 3.3 evidence in the user message. Use **`do_sample=False`, `max_new_tokens=40`**, and decode **only newly generated tokens**, omitting special tokens.

Your system instruction must request a **single short sentence** based solely on evidence. If `uncertain=false`, describe top-1 cautiously (e.g. *appears to show*). If `uncertain=true`, explicitly state uncertainty and present both class predictions as **alternatives**. Prohibit actions, colours, counts, backgrounds, and numeric certainty claims. **Use Qwen's actual output; do not replace it with hard-coded sentences.**

In [ ]:
# Task 4.1 (4 marks)
QWEN_LOAD_SETTINGS = {
    "local_files_only": True,
    "dtype": QWEN_DTYPE,  # for newer Transformers; older versions may require torch_dtype
}
GENERATION_SETTINGS = {
    "do_sample": False,
    "max_new_tokens": 40,
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
try:
    qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **QWEN_LOAD_SETTINGS)
except TypeError:
    qwen_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, local_files_only=True, torch_dtype=QWEN_DTYPE
    )
qwen_model = qwen_model.to(DEVICE)
qwen_model.eval()
for parameter in qwen_model.parameters():
    parameter.requires_grad = False

print("Qwen device:", next(qwen_model.parameters()).device)
print("Qwen dtype:", next(qwen_model.parameters()).dtype)
print("Trainable Qwen parameters:", sum(p.numel() for p in qwen_model.parameters() if p.requires_grad))

In [ ]:
SYSTEM_PROMPT = """You write one short sentence from image-classifier evidence. You cannot see the image.
Use only the class names in the evidence.
If uncertain=false, use only top1 and cautious wording: "The image appears to show a <top1>."
If uncertain=true, explicitly state uncertainty and give top1 and top2 as alternatives: "The image is uncertain, but it may show either a <top1> or a <top2>."
Never mention actions, colours, counts, backgrounds, or scores.
Never make numeric certainty claims.
Output only the sentence."""


def generate_description(evidence):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Evidence: {evidence}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([prompt], return_tensors="pt").to(DEVICE)
    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = qwen_model.generate(
            **model_inputs,
            **GENERATION_SETTINGS,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = generated_ids[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


descriptions = []
for index, evidence in enumerate(evidences):
    description = generate_description(evidence)
    descriptions.append(description)
    print(f"Test image {index}: {description}")

### Task 4.2 - Inspect image–evidence–text consistency (4 marks)

For the **same first five test images**, display: image, true class (for display only), top-two predictions/scores, uncertainty flag, and Qwen sentence. Save results to `summaries.csv`.

For each output, manually check: (1) class wording agrees with evidence, (2) uncertainty phrasing follows the flag, (3) no unsupported visual details. Report three counts **out of five**, and explain one observed failure (or explicitly state none and describe a plausible failure). Distinguish classifier errors from generation errors. **BLEU/ROUGE are not required.**

For consistent grading, use these suggested CSV columns:
`test_index,true_class,top1,score1,top2,score2,uncertain,description,class_consistent,uncertainty_consistent,no_unsupported_details`.
This is an output-format suggestion; it does not change the assessment criteria in the PDF.

In [ ]:
# Task 4.2 (4 marks)
import csv
import re
import textwrap

COLOUR_WORDS = r"red|orange|yellow|green|blue|purple|pink|brown|black|white|grey|gray|silver|gold"
COUNT_WORDS = r"two|three|four|five|six|seven|eight|nine|ten|several|many|few|pair|group"
ACTION_WORDS = r"flying|running|driving|sailing|jumping|sitting|standing|walking|swimming|eating|parked|landing"
BACKGROUND_WORDS = r"sky|road|street|field|grass|water|ocean|sea|forest|background|airport|harbor|harbour"
UNCERTAINTY_WORDS = r"uncertain|unsure|unclear|may|might|either|possibly|perhaps"
CAUTIOUS_WORDS = r"appears?|seems?|looks like|likely"
UNSUPPORTED_PATTERN = rf"\b({COLOUR_WORDS}|{COUNT_WORDS}|{ACTION_WORDS}|{BACKGROUND_WORDS})\b|\d|%"


def contains(pattern, text):
    return re.search(pattern, text, flags=re.IGNORECASE) is not None


def first_pass_review(description, evidence_fields):
    """A first-pass aid only; confirm every result by reading the sentence yourself."""
    mentioned = {name for name in class_names if contains(rf"\b{name}s?\b", description)}
    uncertain = evidence_fields["uncertain"] == "true"
    expected = {evidence_fields["top1"], evidence_fields["top2"]} if uncertain else {evidence_fields["top1"]}
    has_uncertainty = contains(rf"\b({UNCERTAINTY_WORDS})\b", description)
    has_caution = contains(rf"\b({CAUTIOUS_WORDS})\b", description)
    return {
        "class_consistent": mentioned == expected,
        "uncertainty_consistent": has_uncertainty if uncertain else (has_caution and not has_uncertainty),
        "no_unsupported_details": not contains(UNSUPPORTED_PATTERN, description),
    }


# After reading each sentence, record any correction here, for example:
# MANUAL_OVERRIDES = {2: {"no_unsupported_details": False}}
MANUAL_OVERRIDES = {}

CSV_COLUMNS = ["test_index", "true_class", "top1", "score1", "top2", "score2", "uncertain", "description",
               "class_consistent", "uncertainty_consistent", "no_unsupported_details"]
CHECK_COLUMNS = CSV_COLUMNS[-3:]
summary_rows = []

for index, (evidence, description) in enumerate(zip(evidences, descriptions)):
    fields = parse_evidence(evidence)
    true_class = class_names[int(y_test[index])]  # display and analysis only; never sent to Qwen
    review = first_pass_review(description, fields)
    review.update(MANUAL_OVERRIDES.get(index, {}))
    summary_rows.append({
        "test_index": index,
        "true_class": true_class,
        "top1": fields["top1"],
        "score1": fields["score1"],
        "top2": fields["top2"],
        "score2": fields["score2"],
        "uncertain": fields["uncertain"],
        "description": description,
        **review,
    })

    plt.figure(figsize=(3, 3))
    plt.imshow(x_test[index])
    plt.axis("off")
    plt.show()
    print(textwrap.fill(
        f"Image {index} | true={true_class} | top1={fields['top1']} ({fields['score1']}) | "
        f"top2={fields['top2']} ({fields['score2']}) | uncertain={fields['uncertain']}", 100))
    print(textwrap.fill(f"Qwen: {description}", 100))
    print("Checks:", review)
    print("Classifier top-1 correct:", fields["top1"] == true_class, "\n")

SUMMARY_PATH = OUTPUT_DIR / "summaries.csv"
with open(SUMMARY_PATH, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(summary_rows)
print("Saved:", SUMMARY_PATH)

for column in CHECK_COLUMNS:
    print(f"{column}: {sum(row[column] for row in summary_rows)} / {len(summary_rows)}")

print("\nClassifier errors vs generation errors:")
for row in summary_rows:
    failed_checks = [column for column in CHECK_COLUMNS if not row[column]]
    classifier_error = row["top1"] != row["true_class"]
    if classifier_error or failed_checks:
        print(f"Image {row['test_index']}: classifier_error={classifier_error}, generation_failures={failed_checks}")

**Task 4.2 - manual analysis (fill in after running and reading the five sentences):**

- Class wording consistent with evidence: **[count] / 5**
- Uncertainty rule followed: **[count] / 5**
- No unsupported visual detail: **[count] / 5**

**Observed failure or plausible failure; classifier error vs language-generation error:**

> Observed failure: **[image index, evidence, and Qwen sentence]**, which failed because **[reason]**. If all checks pass, state that none was observed; a plausible failure is that Qwen adds "flying" to an airplane prediction, which the evidence does not support.
>
> A classifier error occurs when the ViT predicts the wrong class, even if the sentence faithfully follows the evidence. A generation error occurs when Qwen departs from the evidence by changing a class, ignoring the uncertainty flag, or adding unsupported details, even if the classifier was correct.

### Task 4.3 - Explain the limits (2 marks)

In **at most 100 words**, explain (i) why this pipeline produces only category-level descriptions and (ii) what additional visual supervision or trained multimodal components would be needed for descriptions of attributes, relationships, and objects.

**Task 4.3 written response (≤100 words):**

> The ViT is trained only on single CIFAR-10 class labels, and Qwen receives only two class names and scores, so the pipeline has no evidence beyond category-level identity. Describing attributes, relationships, and multiple objects would need images annotated with captions, attributes, object boxes, or scene relations; a visual encoder trained on them, retaining spatial patch features; and a trained multimodal bridge, such as a projection into text embeddings or cross-attention, jointly trained so the decoder can generate from visual features.